# MACE NLU Fine-Tuning with Unsloth

Fine-tune Qwen2.5-3B or Llama-3.2-3B on MACE NLU training data.

**Requirements:**
- Colab Pro or Kaggle with T4/A100 GPU
- ~8GB VRAM for QLoRA

**Output:** GGUF model for local Ollama deployment

In [ ]:
# Install Unsloth (2x faster fine-tuning, 60% less memory)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# Configuration
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # or "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
LORA_R = 32
LORA_ALPHA = 64
OUTPUT_NAME = "mace-nlu-qwen2.5-3b"

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load base model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

print(f"Loaded {MODEL_NAME}")

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Added LoRA adapters (r={LORA_R}, alpha={LORA_ALPHA})")

In [ ]:
# Upload your instruction_data.jsonl file to Colab
# Or use this cell to create sample data for testing

from google.colab import files
print("Upload instruction_data.jsonl:")
uploaded = files.upload()

TRAINING_FILE = list(uploaded.keys())[0]
print(f"Using: {TRAINING_FILE}")

In [ ]:
# Load and prepare dataset
import json
from datasets import Dataset

# Load JSONL
data = []
with open(TRAINING_FILE, 'r') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"Loaded {len(data)} examples")

# ChatML format template
CHATML_TEMPLATE = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{user}<|im_end|>
<|im_start|>assistant
{assistant}<|im_end|>"""

def format_example(example):
    """Format for ChatML training."""
    if "messages" in example:
        # Already in ChatML format
        msgs = example["messages"]
        system = next((m["content"] for m in msgs if m["role"] == "system"), "")
        user = next((m["content"] for m in msgs if m["role"] == "user"), "")
        assistant = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
    else:
        # Alpaca/standard format
        system = example.get("system", "You are MACE-NLU, a parser for the MACE memory system.")
        user = f"{example.get('instruction', '')}\n{example.get('input', '')}".strip()
        assistant = example.get("output", "")
    
    return {"text": CHATML_TEMPLATE.format(system=system, user=user, assistant=assistant)}

# Convert to Dataset
formatted_data = [format_example(ex) for ex in data]
dataset = Dataset.from_list(formatted_data)

print(f"Dataset ready: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['text'][:500]}...")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training configuration
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,  # Adjust based on dataset size
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

print("Trainer configured. Starting training...")

In [ ]:
# Train!
trainer_stats = trainer.train()

print(f"\n{'='*50}")
print("TRAINING COMPLETE!")
print(f"{'='*50}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# Test the model
FastLanguageModel.for_inference(model)

test_input = "yo remind me 2 get milk tmrw"

messages = [
    {"role": "system", "content": "You are MACE-NLU. Parse user input into MemoryAction JSON."},
    {"role": "user", "content": f"Parse: {test_input}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Input: {test_input}")
print(f"\nOutput:\n{response}")

In [ ]:
# Save LoRA adapters
model.save_pretrained(f"{OUTPUT_NAME}-lora")
tokenizer.save_pretrained(f"{OUTPUT_NAME}-lora")
print(f"Saved LoRA adapters to {OUTPUT_NAME}-lora/")

In [ ]:
# Export to GGUF for Ollama
# This merges LoRA weights and quantizes to GGUF

model.save_pretrained_gguf(
    OUTPUT_NAME,
    tokenizer,
    quantization_method="q4_k_m"  # Good balance of size/quality
)

print(f"\nExported to {OUTPUT_NAME}.gguf")
print("\nTo use in Ollama:")
print("1. Download the .gguf file")
print("2. Create Modelfile:")
print(f"   FROM ./{OUTPUT_NAME}.gguf")
print("3. ollama create mace-nlu -f Modelfile")

In [ ]:
# Download GGUF file
from google.colab import files

import os
gguf_file = f"{OUTPUT_NAME}-unsloth.Q4_K_M.gguf"
if os.path.exists(gguf_file):
    files.download(gguf_file)
    print(f"Downloaded: {gguf_file}")
else:
    print(f"File not found: {gguf_file}")
    print("Available files:")
    !ls -la *.gguf 2>/dev/null || echo "No GGUF files found"

## Deployment Instructions

After downloading the GGUF file:

### 1. Create Ollama Modelfile
```
FROM ./mace-nlu-qwen2.5-3b-unsloth.Q4_K_M.gguf

TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

SYSTEM "You are MACE-NLU, a Natural Language Understanding parser. Parse user input into MemoryAction JSON."

PARAMETER temperature 0.1
PARAMETER stop "<|im_end|>"
```

### 2. Create Model in Ollama
```bash
ollama create mace-nlu -f Modelfile
```

### 3. Test
```bash
ollama run mace-nlu "Parse: remind me to buy milk tomorrow"
```